# Seeing with OpenCV

**OpenCV** (short for "Open Source Computer Vision") is the world's most widely used library for working with images and video. It is the toolkit behind everything from phone camera filters to self-driving cars.

In this notebook you'll learn the basics: how a computer stores a picture, and how to crop, recolor, blur, and draw on one. Then you'll find edges and shapes, and finish by running your code on live video from your webcam.

# Loading an Image

OpenCV is imported under the name `cv2`. `cv2.imread(path)` reads an image file, and `cv2.imshow(name, image)` displays it.

To OpenCV, an image is simply a **NumPy array** of numbers. Its `.shape` is `(height, width, channels)`: the number of rows of pixels, the number of columns, and how many color values each pixel has.

Our photo is large, so we use `cv2.resize` to shrink it to 640 pixels wide and 360 tall. Smaller images are faster to work with.

In [ ]:
import cv2
from codetto import samples

img = cv2.imread(samples.Images.Cat)
print("Original shape:", img.shape)

img = cv2.resize(img, (640, 360))
print("Resized shape:", img.shape)

cv2.imshow("Cat", img)

# Pixels and Colors

Each **pixel** holds three numbers from 0 to 255: how much blue, green, and red it contains. Notice the order. Most software uses RGB, but OpenCV stores colors as **BGR** (blue, green, red). It is the most common surprise for newcomers.

You read a pixel with `img[y, x]`. The row (`y`) comes first, then the column (`x`), and `(0, 0)` is the top-left corner.

You can change pixels the same way. Below, we copy the image and paint a band across the top by setting a whole block of pixels at once. Try changing the color.

In [ ]:
print("Top-left pixel (B, G, R):", img[0, 0])
print("Pixel at row 150, column 400:", img[150, 400])

painted = img.copy()
painted[0:40, :] = (0, 0, 255)  # BGR, so this is red

cv2.imshow("Painted", painted)

## Cropping

Because an image is an array, **cropping** is just slicing. `img[y1:y2, x1:x2]` keeps the rows from `y1` to `y2` and the columns from `x1` to `x2`.

Change the numbers to crop a different part of the photo.

In [ ]:
face = img[20:230, 320:530]
print("Cropped shape:", face.shape)

cv2.imshow("Face", face)

# Grayscale

`cv2.cvtColor` converts an image from one color system to another. The most useful conversion is to **grayscale**, where each pixel is a single brightness value from 0 (black) to 255 (white).

Many computer vision techniques start by converting to grayscale. With one number per pixel instead of three, there is less data to process, and brightness is often all you need.

Look at the shape. The third number has gone, because there is now only one channel.

In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

print("Color shape:", img.shape)
print("Grayscale shape:", gray.shape)
print("Top-left pixel:", gray[0, 0])

cv2.imshow("Grayscale", gray)

# Blurring

A **blur** replaces each pixel with an average of the pixels around it. It is how photo apps soften a background, and it is also used to remove tiny specks and details before looking for larger features.

`cv2.GaussianBlur` takes the image and the size of the neighborhood to average over, called the **kernel**. A bigger kernel means a stronger blur. The size must be an odd number, so that there is a pixel exactly in the middle.

Drag the slider to change the kernel size.

In [ ]:
KERNEL = 15 #@param {type:"slider", min:1, max:51, step:2}

blurred = cv2.GaussianBlur(img, (KERNEL, KERNEL), 0)

cv2.imshow("Blurred", blurred)

# Finding Edges

An **edge** is a place where the brightness changes sharply, such as the outline of an object. Edges let a computer pick out shapes while ignoring color and shading.

`cv2.Canny` is the classic edge detector. It takes a grayscale image and two thresholds, and returns a black image with the edges drawn in white. Lower thresholds find more edges, including faint ones. Higher thresholds keep only the strongest.

Try `cv2.Canny(gray, 30, 80)` and `cv2.Canny(gray, 200, 400)` to see the difference.

In [ ]:
edges = cv2.Canny(gray, 100, 200)

cv2.imshow("Edges", edges)

# Drawing on Images

OpenCV can draw shapes and text straight onto an image. Each function takes the image, a position, a color (in BGR), and a thickness. A thickness of `-1` fills the shape in.

Here we start with a blank image. `np.zeros` makes an array full of zeros, and since 0 means "no color", the result is black. Then we draw three shapes on it.

In [ ]:
import numpy as np

shapes = np.zeros((300, 500, 3), dtype=np.uint8)

cv2.rectangle(shapes, (40, 60), (160, 180), (255, 200, 0), -1)
cv2.circle(shapes, (270, 120), 65, (0, 220, 255), -1)
cv2.rectangle(shapes, (380, 40), (450, 250), (120, 120, 255), -1)
cv2.putText(shapes, "Three shapes", (40, 270),
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (90, 90, 90), 2)

cv2.imshow("Shapes", shapes)

# Finding Shapes

Now let's get the computer to find those shapes by itself. This takes two steps.

1. A **threshold** turns the image into pure black and white. Every pixel brighter than a cutoff becomes white, and everything else becomes black. The result is called a **mask**.
2. `cv2.findContours` traces the outline of each white region in the mask. Each outline is a **contour**.

For each contour, `cv2.boundingRect` gives the smallest upright rectangle that contains it, and `cv2.contourArea` gives its size in pixels. The dim gray text is darker than the cutoff, so it is ignored.

In [ ]:
shapes_gray = cv2.cvtColor(shapes, cv2.COLOR_BGR2GRAY)
_, mask = cv2.threshold(shapes_gray, 100, 255, cv2.THRESH_BINARY)

contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
print("Shapes found:", len(contours))

found = shapes.copy()
for contour in contours:
  x, y, w, h = cv2.boundingRect(contour)
  cv2.rectangle(found, (x, y), (x + w, y + h), (255, 255, 255), 2)
  print("Shape at", (x, y), "with area", int(cv2.contourArea(contour)))

cv2.imshow("Found", found)

# Live Video

A video is just a series of images, called **frames**, shown quickly one after another. Everything you have learned works on video too. You simply do it to every frame.

`cv2.VideoCapture(0)` opens your webcam, and each call to `cap.read()` returns the newest frame. The loop below converts every frame to grayscale, finds its edges, and displays the result.

Your browser will ask for permission to use the camera. The video stays on your computer and is never uploaded. Press the Stop button to end the loop. The `finally` block makes sure the camera is switched off.

In [ ]:
cap = cv2.VideoCapture(0)
try:
  while True:
    ok, frame = cap.read()
    if not ok:
      continue
    frame = cv2.flip(frame, 1)  # mirror, like a selfie camera
    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    cv2.imshow("Live edges", cv2.Canny(frame_gray, 100, 200))
finally:
  cap.release()

Try swapping the edge detector for a blur, or draw a rectangle on each frame before showing it.

OpenCV can also run neural networks that recognize what is in a picture. To see that in action, open [[512C959E-E005-4B30-ACD4-E67A119B9A9D|Face Detection]] or [[809A241E-EEE6-4F90-BFF5-01F2910C5A06|Object Detection]].

# Check Your Understanding